
# Notebook 03 — Create the Baseline Genie Agent

**What you'll learn:** how to create a Genie agent from Unity Catalog tables — the natural-language interface where users ask questions and Genie writes the SQL.

This is the **first of three agents** the workshop builds on the *same* manufacturing data, so we can prove that **curation is what earns trust**:

| Notebook | Agent | What it has |
|---|---|---|
| **03 (here)** | **Baseline** | Tables only, minimal instructions — the *before* |
| **03b** | **Metric View** | Built on a governed metric view (semantic layer) |
| **04** | **Knowledge Store** | Measures, filters, fields, joins, synonyms, example SQL — the *after* |

Notebook **08** runs the same benchmark questions against all three and shows the accuracy difference.

**Why start blank?** So the gain from curation is *measured*, not assumed. A tables-only agent has to guess join paths and business formulas — you'll watch it struggle in 08, then watch curation fix it.

> **Two ways to build — we show both.** This notebook creates the agent **programmatically** so everyone gets the same result by running the same cell. You can also do it in the **UI**: **New → Genie** in the sidebar, pick the tables, **Create**. Both call the same API.

**Before you start:** run notebook **02** (data setup).

**Compute:** Serverless.

## Serverless compute

Attach **Serverless** notebook compute. This notebook uses the Databricks **SDK** and the Genie REST API; `spark` is used only to save agent IDs to the `workshop_config` table.

## Configuration

Catalog and schema come from **notebook 00** (`%run` below). All notebooks share the same values so the tables and agents line up.

In [ ]:
%run ./00_workshop_config

In [ ]:
from databricks.sdk import WorkspaceClient
import re
import json
import uuid
import requests

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id: str) -> str:
    # Clickable Genie UI link: /genie/rooms/<space_id>?o=<workspace_id>
    # (REST APIs use /api/2.0/genie/spaces/... — a different path.)
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

## Pick a SQL warehouse

Genie runs its generated SQL on a **Pro or Serverless** SQL warehouse. This cell picks a running/starting/serverless one automatically.

In [ ]:
warehouses = list(w.warehouses.list())
warehouse_id = None

for wh in warehouses:
    state = str(wh.state).upper() if wh.state else ""
    if state in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        print(f"Using warehouse: {wh.name} ({wh.id}) state={state}")
        break

if not warehouse_id:
    for wh in warehouses:
        if getattr(wh, "enable_serverless_compute", False) or "serverless" in (wh.name or "").lower():
            warehouse_id = wh.id
            print(f"Using serverless warehouse: {wh.name} ({wh.id})")
            break

if not warehouse_id and warehouses:
    wh = warehouses[0]
    warehouse_id = wh.id
    print(f"Using first warehouse: {wh.name} ({wh.id}) state={wh.state}")

if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")

## Build a minimal baseline agent

The baseline gets the **seven tables** and one short instruction — no measures, joins, synonyms, or examples. That comes in 03b and 04.

We use the Genie REST API (`/api/2.0/genie/spaces`) with a `serialized_space` (v2) payload. `create_or_update_genie_space` is **idempotent**: if an agent with the same title already exists, it PATCHes it instead of creating a duplicate.

In [ ]:
# The seven manufacturing tables (sorted — the API expects sorted identifiers).
TABLE_IDENTIFIERS = sorted([
    f"{fqn}.plants",
    f"{fqn}.production_lines",
    f"{fqn}.operators",
    f"{fqn}.production_events",
    f"{fqn}.quality_metrics_daily",
    f"{fqn}.safety_incidents",
    f"{fqn}.equipment_feedback",
])
tables_config = [{"identifier": t} for t in TABLE_IDENTIFIERS]


def build_serialized_baseline():
    """Deliberately minimal: tables + one lean instruction, no curation.

    This is the 'before'. Notebook 04 adds the Knowledge Store that makes an
    agent trustworthy; here we want to see how Genie does with almost nothing.
    """
    baseline_instr = (
        "You answer questions using SQL against the attached tables only. "
        "Use fully qualified table names. "
        "Do not invent business rules or thresholds unless they appear in the data."
    )
    return json.dumps({
        "version": 2,
        "config": {"sample_questions": []},
        "data_sources": {"tables": tables_config},
        "instructions": {
            "text_instructions": [{"id": uuid.uuid4().hex, "content": [baseline_instr]}],
            "example_question_sqls": [],
        },
    })


def list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_or_update_genie_space(title, description, serialized_space_str):
    """Create the agent, or PATCH it if one with this title already exists."""
    for s in list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            pr = requests.patch(
                f"{host}/api/2.0/genie/spaces/{sid}",
                headers=headers,
                json={"title": title, "description": description,
                      "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
            )
            print(f"Updated existing: {title!r} -> {sid} ({pr.status_code})")
            return sid, genie_ui_room_url(sid)
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces", headers=headers,
        json={"title": title, "description": description,
              "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")
    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)

## Create the baseline agent and save its ID

The ID is stored in `workshop_config` under `genie_space_id_blank` so notebook **08** can compare it against the curated and metric-view agents. `save_config_keys` (from notebook 00) upserts, so 03 / 03b / 04 don't clobber each other.

In [ ]:
baseline_id, baseline_url = create_or_update_genie_space(
    GENIE_TITLE_BASELINE, GENIE_DESC_BASELINE, build_serialized_baseline()
)

save_config_keys([
    {"key": CFG_KEY_BASELINE, "value": baseline_id,
     "space_name": GENIE_TITLE_BASELINE, "space_url": baseline_url},
])

print()
print("=" * 70)
print("BASELINE AGENT CREATED")
print("=" * 70)
print(f"  {GENIE_TITLE_BASELINE}")
print(f"  {baseline_url}")
print("=" * 70)
print()
print('Try asking it: "What is the average OEE by plant for 2024?"')
print("With no curated joins or measures it may pick the wrong table or")
print("formula — that is exactly the gap 03b and 04 close.")

In [ ]:
import html

for _r in spark.sql(
    f"SELECT key, value, space_name FROM {fqn}.workshop_config ORDER BY key"
).collect():
    _u = genie_ui_room_url(_r["value"])
    displayHTML(
        "<p><b>" + html.escape(str(_r["key"])) + "</b> &mdash; "
        + '<a href="' + html.escape(_u, quote=True)
        + '" target="_blank" rel="noopener">' + html.escape(str(_r["space_name"]))
        + '</a><br/><code style="font-size:11px">' + html.escape(_u) + "</code></p>"
    )

## Next

- **03b — Metric Views:** build a governed semantic layer, then a metric-view-based agent.
- **04 — Knowledge Store:** curate this data with measures, filters, fields, joins, synonyms, and example SQL — this becomes the **primary** agent your users trust.

Then **08** compares Baseline vs. Metric View vs. Knowledge Store head-to-head.